# PS-S06E04: Model Stacking


This notebook tackles the [**Playground Series – Season 6, Episode 4: Predicting Irrigation Need**](https://www.kaggle.com/competitions/playground-series-s6e4), a competition focused on predicting the irrigation needs of crops. The goal is to classify the required irrigation level into three categories: **Low (0), Medium (1), or High (2)**, based on environmental features including soil moisture, temperature, humidity, and crop variety.

## Ensemble Strategy: Stacking with a Logistic Meta-Model

Blending (weighted averaging) can fail when one model is dominant or when models are highly correlated. Stacking treats ensembling as a supervised learning problem: we train a small meta-model to combine base model probabilities.

A practical advantage is that a weaker model can still help if it is *differently wrong*—the meta-model can learn when to trust it.


## Install Needed Packages

In [1]:
import os
import glob
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV

from ps_s06e04_experiment_setup import ExperimentSetup

warnings.filterwarnings('ignore')

%matplotlib inline


In [2]:
helper = ExperimentSetup()

seed = helper.set_seeds()

helper.configure_pandas()
helper.suppress_warnings()

TARGET = 'Irrigation_Need'


Random seed set to: 10301
Warnings suppressed.


In [3]:
# Load ground truth labels
training_df = helper.read_dataset('training')

# Encode target labels
target_mapping = {'Low': 0, 'Medium': 1, 'High': 2}
y_true = training_df[TARGET].map(target_mapping).astype(int)


TRAINING DATASET

   id Soil_Type  Soil_pH  Soil_Moisture  Organic_Carbon  \
0   0     Loamy    4.920         32.580           1.010   
1   1      Clay    7.080         56.610           0.440   
2   2      Clay    5.690         27.710           0.810   
3   3     Sandy    5.650         13.320           1.330   
4   4      Clay    7.960         59.140           0.380   

   Electrical_Conductivity  Temperature_C  Humidity  Rainfall_mm  \
0                    3.050         15.010    50.610      725.990   
1                    2.000         22.920    67.860      985.660   
2                    2.830         26.970    92.220    2,201.700   
3                    0.870         13.320    61.570    1,357.330   
4                    0.960         20.220    91.110    1,538.200   

   Sunlight_Hours  Wind_Speed_kmh  Crop_Type Crop_Growth_Stage  Season  \
0           5.900          16.790  Sugarcane            Sowing    Zaid   
1           6.980           3.390      Wheat        Vegetative  Kharif

In [4]:
submission_df = helper.read_dataset('submission')

## Loading OOF and Test Probabilities

We load the out-of-fold (OOF) and test-set **probabilities** generated by individual model notebooks.

Expected file naming:
- `*_oof_probs.csv` contains columns: `id`, one probability column (e.g. `prob_xgb`), and optionally `target`
- `*_test_probs.csv` contains columns: `id` and one probability column

All files should share the same `id` values as the competition datasets.


In [5]:
oof_files = []
test_files = []

# Local convention: predictions stored under predictions/s06e04/
# Kaggle convention: mount dataset input that contains predictions/
if helper.running_in_kaggle():
    pred_dir = '/kaggle/input/notebooks/stephentarter/ps-s06e04-*/predictions'
else:
    pred_dir = 'predictions'

oof_files = sorted(glob.glob(f'{pred_dir}/*_oof_probs.csv'))
test_files = sorted(glob.glob(f'{pred_dir}/*_test_probs.csv'))

print(f'Found {len(oof_files)} OOF files and {len(test_files)} Test files.')
if len(oof_files) > 0:
    print('OOF files:')
    for f in oof_files:
        print(' -', os.path.basename(f))
if len(test_files) > 0:
    print('Test files:')
    for f in test_files:
        print(' -', os.path.basename(f))


Found 3 OOF files and 3 Test files.
OOF files:
 - catboost_oof_probs.csv
 - lgb_oof_probs.csv
 - xgb_oof_probs.csv
Test files:
 - catboost_test_probs.csv
 - lgb_test_probs.csv
 - xgb_test_probs.csv


In [6]:
# Helper to load and merge probabilities
def load_probs(file_list, index_col='id'):
    df_list = []
    for file in file_list:
        base = os.path.basename(file)
        model_name = (
            base.replace('_oof_probs.csv', '')
                .replace('_test_probs.csv', '')
        )

        df = pd.read_csv(file)

        # Identify the probability column (exclude id/target)
        ignore = {'id', 'target', TARGET}
        prob_cols = [c for c in df.columns if c not in ignore]
        
        # Rename columns to include model name to avoid collisions
        # e.g., 'catboost_prob_low', 'catboost_prob_medium', etc.
        rename_map = {c: f"{model_name}_{c}" for c in prob_cols}

        # Set index and keep only the renamed prob columns
        subset = df.set_index(index_col).rename(columns=rename_map)[list(rename_map.values())]
        df_list.append(subset)

    # Join all models side-by-side
    return pd.concat(df_list, axis=1)


In [7]:
# Create DataFrames
oof_df = load_probs(oof_files)
test_df = load_probs(test_files)

# Look for missing rows in test data
missing_by_model = test_df.isna().sum().sort_values(ascending=False)
print("Missing test rows per model:")
print(missing_by_model[missing_by_model > 0])

if missing_by_model.any():
    bad = missing_by_model[missing_by_model > 0].index.tolist()
    print("\nFirst few missing ids for each bad model:")
    for m in bad:
        print(m, test_df.index[test_df[m].isna()][:10].tolist())

# Create an id-indexed label series (0/1)
y_by_id = pd.Series(y_true.values, index=training_df['id'].values)

# Keep only rows where all models have OOF probabilities
oof_df = oof_df.sort_index().dropna(axis=0)
y_true_aligned = y_by_id.loc[oof_df.index].values

# Align test rows by id as well
test_df = test_df.sort_index()

print(f'OOF Shape: {oof_df.shape}')
print(f'Test Shape: {test_df.shape}')


Missing test rows per model:
Series([], dtype: int64)
OOF Shape: (630000, 9)
Test Shape: (270000, 9)


In [8]:
print('\nIndividual Model Balanced Accuracy (OOF):')
individual_scores = {}

# Group columns by model to evaluate them
model_names = list(set([c.split('_prob_')[0] for c in oof_df.columns]))

for model in model_names:
    cols = [f"{model}_prob_low", f"{model}_prob_medium", f"{model}_prob_high"]
    # Get hard labels via argmax
    preds = np.argmax(oof_df[cols].values, axis=1)
    score = balanced_accuracy_score(y_true_aligned, preds)
    individual_scores[model] = score
    print(f'{model:10}: {score:.6f}')

best_single_model = max(individual_scores, key=individual_scores.get)
print(f'\nBest single model: {best_single_model} ({individual_scores[best_single_model]:.6f})')


Individual Model Balanced Accuracy (OOF):
catboost  : 0.969854
xgb       : 0.972052
lgb       : 0.969262

Best single model: xgb (0.972052)


## Stacking Optimization: Logistic Regression Meta-Model

We train a second-level meta-model on the base models' **OOF probabilities**.

* **Features (X):** OOF probabilities from each base model.
* **Target (y):** true label (0=Absence, 1=Presence).

We use **logistic regression with L2 regularization** as the meta-model. It is fast, stable under correlation, and outputs a valid probability in [0, 1].
The regularization strength is tuned via internal cross-validation.


In [17]:
# ==========================================
# STACKING STRATEGY (Meta-Model)
# ==========================================

print('Preparing stacking data...')

model_cols = list(oof_df.columns)
print('Stacking models:', model_cols)

X_stack = oof_df[model_cols].values
y_stack = y_true_aligned
X_test_stack = test_df.values

# Standardize meta-features (helps logistic regression)
scaler = StandardScaler()
X_stack_s = scaler.fit_transform(X_stack)
X_test_stack_s = scaler.transform(X_test_stack)

# Multi-class Logistic Regression Stacker
meta_model = LogisticRegressionCV(
    Cs=20,
    cv=5,
    scoring='balanced_accuracy',
    multi_class='multinomial',
    penalty='l2',
    max_iter=5000,
    n_jobs=-1,
    random_state=seed,
)

meta_model.fit(X_stack_s, y_stack)

oof_meta_preds = meta_model.predict(X_stack_s)
score_meta = balanced_accuracy_score(y_stack, oof_meta_preds)
print(f'Meta-model OOF Balanced Accuracy: {score_meta:.6f}')


Preparing stacking data...
Stacking models: ['catboost_prob_low', 'catboost_prob_medium', 'catboost_prob_high', 'lgb_prob_low', 'lgb_prob_medium', 'lgb_prob_high', 'xgb_prob_low', 'xgb_prob_medium', 'xgb_prob_high']
Meta-model OOF Balanced Accuracy: 0.969266


## Final Ensemble & Submission

We generate test-set probabilities by applying the trained meta-model to the matrix of base-model test probabilities.
The submission file uses the competition's sample submission schema.


In [18]:
# Generate test probabilities from the stacker
test_meta_prob = meta_model.predict(X_test_stack_s)

# Fill submission using sample_submission column name
sub = submission_df.copy()
target_col = [c for c in sub.columns if c != 'id'][0]
sub[target_col] = test_meta_prob
reverse_mapping = {v: k for k, v in target_mapping.items()}
sub[target_col] = sub[target_col].map(reverse_mapping)

sub.to_csv('submission.csv', index=False)
print('Saved: submission.csv')

print('\nSUBMISSION')
print('==========')
print(sub.head(10))


Saved: submission.csv

SUBMISSION
       id Irrigation_Need
0  630000             Low
1  630001             Low
2  630002             Low
3  630003             Low
4  630004             Low
5  630005          Medium
6  630006             Low
7  630007          Medium
8  630008          Medium
9  630009             Low
